# 🏛️ Deep Modules
## A Philosophy of Software Design for LLMs

John Ousterhout's *A Philosophy of Software Design* defines a **deep module** as one with a simple interface that hides significant internal complexity. The best modules are *deep*: easy to use, powerful inside.

In DSPy, all modules share the same interface:

```python
module(**inputs) → prediction
```

But internally, they range from a single LLM call to a multi-step reasoning loop with tool use.

This notebook explores three levels of depth:

| Module | Depth | What happens inside |
|--------|-------|--------------------|
| `dspy.Predict` | Shallow | 1 LLM call, direct answer |
| `dspy.ChainOfThought` | Medium | 1 LLM call, adds reasoning field |
| `dspy.ReAct` | Deep | N LLM calls + tool use loop |

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy
import ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task
from dspy_tasks.data import TranslateEnDe, SolveMath
from dspy_tasks.actions import run_baseline
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
display(model_dd)

## The Three Depths

All three modules are called the same way — but what happens *inside* is radically different.

```
┌──────────────────────────────────────────────────────────┐
│           All three have the SAME interface:             │
│           module(**inputs) → prediction                  │
├──────────────────────────────────────────────────────────┤
│ Predict        │ 1 LLM call, direct answer              │
│ ChainOfThought │ 1 LLM call, adds reasoning field       │
│ ReAct          │ N LLM calls + tool use loop             │
└──────────────────────────────────────────────────────────┘
```

This is the deep module principle in action: **the interface stays constant while the implementation grows in sophistication**.

In [ ]:
configure_dspy(model=model_dd.value)

# SHALLOW: Just predict
shallow = dspy.Predict(TranslateEnDe)
result_shallow = shallow(english_text="The cat sat on the mat while the dog chased its tail.")
print("Predict (shallow):")
print(f"  → {result_shallow.german_text}\n")

# DEEP: Chain of Thought
deep = dspy.ChainOfThought(TranslateEnDe)
result_deep = deep(english_text="The cat sat on the mat while the dog chased its tail.")
print("ChainOfThought (deep):")
print(f"  Reasoning: {result_deep.rationale[:200] if hasattr(result_deep, 'rationale') else 'N/A'}")
print(f"  → {result_deep.german_text}")

## Where Depth Matters Most

For simple tasks like translation, `Predict` often does fine — the task doesn't *require* reasoning.

But for tasks that need **multi-step reasoning** (math, logic, planning), `ChainOfThought` is transformative. The added reasoning field forces the model to *show its work*, dramatically improving accuracy.

Let's prove it with a math task.

In [ ]:
# Run math task with both module types
task = get_task("math_word")
_, devset = task.split_examples()
eval_set = devset[:8]

from dspy_tasks.calculations import numeric_match
from dspy_tasks.actions import _evaluate_examples, _mean

# Shallow
shallow_math = dspy.Predict(task.signature_class)
shallow_results = _evaluate_examples(shallow_math, eval_set, task.metric_fn)
shallow_score = _mean([r["score"] for r in shallow_results])

# Deep
deep_math = dspy.ChainOfThought(task.signature_class)
deep_results = _evaluate_examples(deep_math, eval_set, task.metric_fn)
deep_score = _mean([r["score"] for r in deep_results])

display_score("Predict (shallow)", shallow_score)
display_score("ChainOfThought (deep)", deep_score)
display_improvement(shallow_score, deep_score)

display_insight("Deep Module Lesson",
    f"Same interface, same task, same model — but ChainOfThought scored "
    f"{deep_score:.0%} vs Predict's {shallow_score:.0%}. "
    "The module's internal depth is what makes reasoning possible.")

In [ ]:
from dspy_tasks.config import configure_dspy
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task(tid) for tid in ["translation", "format_compliance", "math_word", "logical_deduction"]]],
    description="Task:")
module_dd = widgets.Dropdown(options=["Predict", "ChainOfThought"], description="Module:")
btn = run_button("Run")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        task = get_task(task_dd.value)
        configure_dspy(model=model_dd.value)
        if module_dd.value == "ChainOfThought":
            module = dspy.ChainOfThought(task.signature_class)
        else:
            module = dspy.Predict(task.signature_class)
        _, devset = task.split_examples()
        results = _evaluate_examples(module, devset[:8], task.metric_fn)
        score = _mean([r["score"] for r in results])
        display_score(f"{task.name} ({module_dd.value})", score)
        display_results_table(results[:5])

btn.on_click(on_run)
display(widgets.HBox([task_dd, module_dd, model_dd, btn]), out)

## Summary

**Deep modules hide complexity.** DSPy's genius is that `Predict`, `ChainOfThought`, and `ReAct` all share the same interface — you choose the depth your task needs.

| Principle | Application |
|-----------|------------|
| Simple interface | `module(**inputs) → prediction` everywhere |
| Choose depth for the task | Translation → `Predict`; Math → `ChainOfThought` |
| Composability | Swap modules without changing any other code |

**Next: How do you *KNOW* which depth is better? → Evaluation as Specification** (Notebook 03)